In [ ]:
# Cell 1: Load training data from CSV (or fallback sample data)
import pandas as pd
from io import StringIO
from pathlib import Path

file_path = Path("data/training_data.csv")

sample_csv = """Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
Sunny,Warm,Normal,Strong,Warm,Same,Yes
Sunny,Warm,High,Strong,Warm,Same,Yes
Rainy,Cold,High,Strong,Warm,Change,No
Sunny,Warm,High,Strong,Cool,Change,Yes
"""

if file_path.exists():
    data = pd.read_csv(file_path)
    print(f"Loaded dataset from: {file_path.resolve()}")
else:
    data = pd.read_csv(StringIO(sample_csv))
    print("Local CSV not found. Using built-in sample training data.")

print(data)

Training data:
   Sunny  Warm Normal  Strong Warm.1    Same  Yes
0  Sunny  Warm   High  Strong   Warm    Same  Yes
1  Rainy  Cold   High  Strong   Warm  Change   No
2  Sunny  Warm   High  Strong   Cool  Change  Yes

Initial specific hypothesis: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']

The final specific hypothesis is: ['Sunny', 'Warm', 'High', 'Strong', '?', '?']


In [ ]:
# Cell 2: Separate input attributes and target labels
attributes = data.iloc[:, :-1]
target = data.iloc[:, -1]

print("Attributes shape:", attributes.shape)
print("Target distribution:")
print(target.value_counts())

In [ ]:
# Cell 3: Define Find-S to learn the most specific consistent hypothesis
def find_s_algorithm(attributes_df, target_series, positive_label="Yes"):
    positive_indices = target_series[target_series == positive_label].index
    if len(positive_indices) == 0:
        raise ValueError(f"No positive examples found for label '{positive_label}'.")

    # Initialize hypothesis with the first positive training example
    hypothesis = list(attributes_df.iloc[positive_indices[0]])

    # Generalize only when another positive example disagrees on an attribute
    for i in positive_indices:
        row = attributes_df.iloc[i]
        for j in range(len(hypothesis)):
            if row.iloc[j] != hypothesis[j]:
                hypothesis[j] = "?"

    return hypothesis

In [ ]:
# Cell 4: Train Find-S and print final hypothesis
final_hypothesis = find_s_algorithm(attributes, target, positive_label="Yes")
print("Final specific hypothesis:", final_hypothesis)

In [ ]:
# Cell 5: Present hypothesis values feature by feature
hypothesis_df = pd.DataFrame({
    "Feature": attributes.columns,
    "Hypothesis Value": final_hypothesis,
})

print(hypothesis_df)

In [ ]:
# Cell 6: Validate coverage of positive examples under learned hypothesis
def matches_hypothesis(example, hypothesis):
    for value, hyp in zip(example, hypothesis):
        if hyp != "?" and value != hyp:
            return False
    return True

positive_examples = attributes[target == "Yes"]
match_results = positive_examples.apply(lambda row: matches_hypothesis(row.values, final_hypothesis), axis=1)

print("Positive examples covered by hypothesis:")
print(match_results.value_counts())